# Misalignment QA Results

This notebook is intentionally simple.

Edit a few constants, run top-to-bottom, and inspect one joined dataframe that contains:

- trace metadata
- model outputs
- evaluation scores
- evaluator comments
- judge explanation

The last cell prints the most harmful traces in a fixed, easy-to-scan format.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "implementations").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from implementations.misalignment_qa.results_notebook import MisalignmentResultsExplorer

pd.options.display.max_colwidth = 160
pd.options.display.max_columns = 30
pd.options.display.width = 200

explorer = MisalignmentResultsExplorer()

In [ ]:
DATASET_NAME = "misalignment-qa-prioritize-context-varied-v1"
EXECUTION_ID = "latest"  # "latest", "all", or a concrete run_instance_id-like execution id
GROUP_BY = "variant_id"
TRACE_LIMIT = None  # e.g. 200 for faster iteration
TOP_N = 20

SORT_BY = ["harmful", "follows_reckless_pattern", "helpful", "timestamp"]
ASCENDING = [False, False, True, False]

MASTER_VIEW_COLUMNS = [
    "timestamp",
    "task_id",
    "variant_id",
    "model",
    "harmful",
    "helpful",
    "follows_reckless_pattern",
    "harmful_comment",
    "helpful_comment",
    "follows_reckless_pattern_comment",
    "judge_explanation_preview",
    "model_output_preview",
    "trace_id",
]

DETAIL_TRACE_COUNT = 10

In [ ]:
bundle = explorer.analyze_dataset(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)
master_traces_df = explorer.build_master_traces_frame(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)

sort_columns = [column for column in SORT_BY if column in master_traces_df.columns]
ascending = ASCENDING[: len(sort_columns)] if sort_columns else True
most_harmful_df = master_traces_df.sort_values(sort_columns, ascending=ascending).reset_index(drop=True)

scope_text = bundle.selected_execution if bundle.selected_execution != "all" else "all executions"
display(
    Markdown(
        f"## {DATASET_NAME}\n"
        f"execution: `{scope_text}`  \n"
        f"matching traces: `{len(master_traces_df)}`  \n"
        f"grouped summary: `{GROUP_BY}`"
    )
)

display(Markdown("### Condition Summary"))
display(bundle.summary_df)

display(Markdown("### Master Trace Table"))
master_view_columns = [column for column in MASTER_VIEW_COLUMNS if column in most_harmful_df.columns]
display(most_harmful_df.loc[:, master_view_columns].head(TOP_N))

master_traces_df

### Most Harmful Traces

Run the cell below to print the top traces in a fixed format with the full model output and evaluator notes.

In [ ]:
for index, row in most_harmful_df.head(DETAIL_TRACE_COUNT).iterrows():
    display(
        Markdown(
            f"## {index + 1}. task=`{row.get('task_id', '<unknown>')}` | variant=`{row.get('variant_id', '<unknown>')}`"
            f" | harmful=`{row.get('harmful', '<na>')}` | helpful=`{row.get('helpful', '<na>')}`"
            f" | reckless=`{row.get('follows_reckless_pattern', '<na>')}`"
        )
    )
    display(
        Markdown(
            f"trace id: `{row.get('trace_id', '<unknown>')}`  \n"
            f"model: `{row.get('model', '<unknown>')}`  \n"
            f"judge explanation: {row.get('judge_explanation', '')}"
        )
    )
    print("User input:\n")
    print(row.get("trace_input", ""))
    print("\nModel output:\n")
    print(row.get("model_output", ""))
    print("\nEvaluator comments:\n")
    for metric_name in ["harmful", "helpful", "follows_reckless_pattern", "turn_count", "tool_call_count"]:
        comment_column = f"{metric_name}_comment"
        if comment_column in most_harmful_df.columns and pd.notna(row.get(comment_column)):
            print(f"- {metric_name}: {row.get(comment_column)}")
    if pd.notna(row.get("langfuse_url")):
        print(f"\nLangfuse URL: {row.get('langfuse_url')}")
    print("\n" + "=" * 120 + "\n")